# Embeddings Setup: Ollama (Local)

This notebook helps you set up and validate the embeddings provider for this repo's hybrid search.

**Why this matters:** hybrid search needs a vector embedding for every chunk indexed and every query
asked. This repo uses **Ollama**, which runs the embedding model locally — no API key, no
document/query text ever leaves your machine.

## What this notebook does

1. Pulls all three supported Ollama embedding models: `nomic-embed-text`, `mxbai-embed-large`, `bge-m3`
2. Lets you select **one** of them to actually use (only one is active at a time)
3. Validates the selection end-to-end:
   - Embeds a sample sentence and checks the vector dimension
   - Indexes it into the real OpenSearch instance and searches it back
   - Confirms the embedding-model label check (added for this repo) passes for the selected model

## Model comparison

| Model | Dimensions | Notes |
|---|---|---|
| `nomic-embed-text` | 768 | Fast, solid general-purpose quality |
| `mxbai-embed-large` | 1024 | English-focused, matches this repo's OpenSearch mapping dimension |
| `bge-m3` | 1024 | **Multilingual**, matches this repo's mapping dimension — recommended default |

**Recommendation:** `bge-m3` is a strong default if you want multilingual support, since it's also
1024-dim (no OpenSearch mapping change needed). `mxbai-embed-large` is a good English-only alternative
at the same dimension. `nomic-embed-text` is smaller/faster but needs a mapping dimension change (768)
if you want to use it.

**Prerequisites:** the full stack running (`docker compose up -d`), including the `ollama` and
`opensearch` containers.

## 1. Environment Setup and Health Check

In [1]:
import sys
import os
from pathlib import Path
import requests

# Find project root and add to Python path
current_dir = Path.cwd()
if current_dir.name == "week4" and current_dir.parent.name == "notebooks":
    project_root = current_dir.parent.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent.parent

sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

os.environ["OPENSEARCH__HOST"] = "http://127.0.0.1:9200"
OLLAMA_URL = "http://127.0.0.1:11434"

print("\nHEALTH CHECK")
print("=" * 40)

all_healthy = True
for name, url in {
    "OpenSearch": "http://127.0.0.1:9200/_cluster/health",
    "Ollama": f"{OLLAMA_URL}/api/version",
}.items():
    try:
        r = requests.get(url, timeout=5)
        print(f"{'✓' if r.status_code == 200 else '✗'} {name}: {r.status_code}")
        all_healthy = all_healthy and r.status_code == 200
    except Exception as e:
        print(f"✗ {name}: {type(e).__name__} - is the stack running? `docker compose up -d`")
        all_healthy = False

if all_healthy:
    print("\n✓ Ready to proceed.")
else:
    print("\n✗ Fix the above before continuing.")

Project root: C:\Users\piete\Repos\pkuppens\production-agentic-rag-course

HEALTH CHECK
✓ OpenSearch: 200
✓ Ollama: 200

✓ Ready to proceed.


## 2. Pull All Three Supported Ollama Embedding Models

This downloads all three models so you can compare/switch freely, even though only one will be
**selected** (configured as active) in step 3. Each is a few hundred MB to ~1.2GB.

In [2]:
import json

SUPPORTED_MODELS = ["nomic-embed-text", "mxbai-embed-large", "bge-m3"]


def pull_model(model: str) -> bool:
    """Pull an Ollama model via the HTTP API, streaming progress."""
    print(f"\nPulling '{model}'...")
    try:
        with requests.post(f"{OLLAMA_URL}/api/pull", json={"model": model}, stream=True, timeout=600) as resp:
            resp.raise_for_status()
            last_status = None
            for line in resp.iter_lines():
                if not line:
                    continue
                event = json.loads(line)
                status = event.get("status")
                if status != last_status:
                    print(f"  {status}")
                    last_status = status
                if event.get("error"):
                    print(f"  ✗ {event['error']}")
                    return False
        print(f"✓ '{model}' ready")
        return True
    except Exception as e:
        print(f"✗ Failed to pull '{model}': {e}")
        return False


print("PULLING ALL SUPPORTED EMBEDDING MODELS")
print("=" * 50)

pull_results = {model: pull_model(model) for model in SUPPORTED_MODELS}

print("\nSummary:")
for model, ok in pull_results.items():
    print(f"  {'✓' if ok else '✗'} {model}")

PULLING ALL SUPPORTED EMBEDDING MODELS

Pulling 'nomic-embed-text'...
  pulling manifest


  pulling 970aa74c0a90


  pulling c71d239df917


  pulling ce4a164fc046


  pulling 31df23ea7daa


...

  pulling b837481ff855


  pulling 38badd946f91


  verifying sha256 digest


  writing manifest
  success
✓ 'mxbai-embed-large' ready

Pulling 'bge-m3'...
  pulling manifest


  pulling daec91ffb5dd
  pulling a406579cd136
  pulling 0c4c9c2a325f
  verifying sha256 digest
  writing manifest
  success
✓ 'bge-m3' ready

Summary:
  ✓ nomic-embed-text
  ✓ mxbai-embed-large
  ✓ bge-m3


## 3. Select One Model to Use

Change `SELECTED_MODEL` below to whichever model you want active, then apply it to your `.env` file
(or set it directly for this notebook session). Only the selected model is used by
`EMBEDDINGS__OLLAMA_EMBEDDING_MODEL` — the other two stay pulled but unused.

In [3]:
# Change this to select which model is active for this session.
SELECTED_MODEL = "bge-m3"  # nomic-embed-text | mxbai-embed-large | bge-m3

assert SELECTED_MODEL in SUPPORTED_MODELS, f"{SELECTED_MODEL!r} is not one of {SUPPORTED_MODELS}"
assert pull_results.get(SELECTED_MODEL), f"'{SELECTED_MODEL}' was not pulled successfully - re-run step 2"

os.environ["EMBEDDINGS__PROVIDER"] = "ollama"
os.environ["EMBEDDINGS__OLLAMA_EMBEDDING_MODEL"] = SELECTED_MODEL

print(f"Selected model for this session: {SELECTED_MODEL}")
print("\nTo make this permanent, set in your .env file:")
print("  EMBEDDINGS__PROVIDER=ollama")
print(f"  EMBEDDINGS__OLLAMA_EMBEDDING_MODEL={SELECTED_MODEL}")
print("\nThen restart the api/airflow containers: docker compose up -d api airflow")

Selected model for this session: bge-m3

To make this permanent, set in your .env file:
  EMBEDDINGS__PROVIDER=ollama
  EMBEDDINGS__OLLAMA_EMBEDDING_MODEL=bge-m3

Then restart the api/airflow containers: docker compose up -d api airflow


## 4. Validate: Generate a Sample Embedding

In [4]:
from src.config import Settings
from src.services.embeddings.factory import make_embeddings_client

settings = Settings()
print(f"Configured provider: {settings.embeddings.provider}")
print(f"Configured model: {settings.embeddings.ollama_embedding_model}")

embeddings_client = make_embeddings_client(settings)
print(f"Client model label: {embeddings_client.model_label}")

sample_text = "Self-attention allows transformers to weigh the importance of different tokens."
sample_embedding = await embeddings_client.embed_query(sample_text)

print(f"\n✓ Generated embedding for sample text")
print(f"  Dimension: {len(sample_embedding)}")
print(f"  Preview: {sample_embedding[:5]}")

expected_dim = settings.opensearch.vector_dimension
assert len(sample_embedding) == expected_dim, (
    f"Embedding dimension {len(sample_embedding)} != OpenSearch mapping dimension {expected_dim}. "
    f"'{SELECTED_MODEL}' is not compatible with this repo's index without a mapping change."
)
print(f"✓ Dimension matches OpenSearch mapping ({expected_dim})")

Configured provider: ollama


Configured model: bge-m3
Client model label: ollama:bge-m3



✓ Generated embedding for sample text
  Dimension: 1024
  Preview: [0.0022862197, -0.01470253, -0.041474648, 0.022505563, -0.020207167]
✓ Dimension matches OpenSearch mapping (1024)


## 5. Validate: Index and Hybrid-Search a Real Chunk

This indexes one temporary chunk into the real OpenSearch instance using the selected model, then
searches it back via hybrid search — proving the full write→read path works, not just the raw
embedding call. The temporary chunk is deleted at the end.

In [5]:
from src.services.opensearch.client import OpenSearchClient

opensearch_client = OpenSearchClient(host="http://127.0.0.1:9200", settings=settings)
TEST_ARXIV_ID = "test-embeddings-setup-notebook"

try:
    doc = {
        "arxiv_id": TEST_ARXIV_ID,
        "chunk_id": f"{TEST_ARXIV_ID}-0",
        "chunk_text": sample_text,
        "chunk_index": 0,
        "title": "Embeddings Setup Validation",
        "authors": "Notebook",
        "section_title": "Introduction",
        "embedding": sample_embedding,
        "embedding_model": embeddings_client.model_label,
    }
    opensearch_client.client.index(index=opensearch_client.index_name, body=doc, refresh=True)
    print(f"✓ Indexed test chunk with label '{embeddings_client.model_label}'")

    query_embedding = await embeddings_client.embed_query("what allows transformers to weigh token importance?")
    results = opensearch_client.search_unified(
        query="self-attention transformers", query_embedding=query_embedding, size=3, use_hybrid=True
    )
    hits = results.get("hits", [])
    found = any(h.get("arxiv_id") == TEST_ARXIV_ID for h in hits)

    print(f"\nHybrid search returned {len(hits)} hit(s)")
    assert found, "Test chunk was not found by hybrid search - something is wrong with the embed→index→search path"
    print("✓ Test chunk found via hybrid search - full round trip works")

finally:
    opensearch_client.client.delete_by_query(
        index=opensearch_client.index_name,
        body={"query": {"term": {"arxiv_id": TEST_ARXIV_ID}}},
        refresh=True,
    )
    print("✓ Cleaned up test chunk")

✓ Indexed test chunk with label 'ollama:bge-m3'



Hybrid search returned 1 hit(s)
✓ Test chunk found via hybrid search - full round trip works
✓ Cleaned up test chunk


## 6. Validate: Embedding-Model Label Check

This repo refuses to mix vectors from different embedding models in the same index. Confirm the
selected model's label passes validation against your real, already-indexed data — if you've
previously indexed papers with a different provider/model, this will correctly raise an error
telling you to reindex.

In [6]:
from src.exceptions import ConfigurationError

indexed_models = opensearch_client.get_indexed_embedding_models()
print(f"Embedding models currently in the index: {indexed_models or '(index empty)'}")

try:
    opensearch_client.validate_embedding_model_consistency(embeddings_client.model_label)
    print(f"\n✓ '{embeddings_client.model_label}' is consistent with the index - safe to use as-is.")
except ConfigurationError as e:
    print(f"\n✗ Mismatch detected:\n{e}")
    print("\nTo fix: re-run the Airflow 'arxiv_paper_ingestion' DAG with replace_existing=True,")
    print("or clear the index, before using this model for real queries.")

print("\n" + "=" * 50)
print(f"SETUP COMPLETE - active provider: ollama, model: {SELECTED_MODEL}")
print("=" * 50)

Embedding models currently in the index: (index empty)

✓ 'ollama:bge-m3' is consistent with the index - safe to use as-is.

SETUP COMPLETE - active provider: ollama, model: bge-m3
